# OCR + NLP Digital Library Pipeline

This notebook contains the main experimental pipeline from my final-year Applied Data Science capstone project. It processes scanned news documents with OpenCV, Tesseract OCR and DeepSeek-OCR, then applies Qwen2.5-0.5B-Instruct for summarisation, classification, and title/author extraction.

**Pipeline:** image preprocessing -> Tesseract OCR -> confidence/loop check -> DeepSeek-OCR fallback -> text quality filtering -> Qwen NLP processing -> CSV output.

> Developed in Google Colab. Update the dataset path in the configuration section before running the notebook in another environment.


In [ ]:
# Install dependencies used by the OCR stage.
!pip install -q pytesseract opencv-python pillow \
transformers==4.46.3 tokenizers==0.20.3 accelerate addict easydict einops

import cv2
import os
import glob
import time
import re
import csv
import numpy as np
import pytesseract
import torch
from collections import Counter
from transformers import AutoModel, AutoTokenizer


## 1. Image Preprocessing and Rotation Correction

Prepare scanned images before OCR using rotation correction, CLAHE contrast enhancement, and Otsu thresholding.


In [ ]:
def correct_rotation(image):
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    coords = cv2.findNonZero(gray)

    if coords is None:
        return image

    angle = cv2.minAreaRect(coords)[-1]

    if angle < -45:
        angle = -(90 + angle)
    else:
        angle = -angle

    (h, w) = image.shape[:2]
    M = cv2.getRotationMatrix2D((w//2, h//2), angle, 1.0)
    return cv2.warpAffine(image, M, (w, h))


def preprocess_pipeline(bgr):
    gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)

    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    gray = clahe.apply(gray)

    _, binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    return bgr, binary

## 2. Tesseract OCR

Run Tesseract and calculate an average word-level confidence score for use in the hybrid OCR decision.


In [ ]:
def run_tesseract(img):
    data = pytesseract.image_to_data(img, output_type=pytesseract.Output.DICT)

    text = " ".join(data["text"])

    confs = [int(c) for c in data["conf"] if c != '-1']
    conf = sum(confs) / max(1, len(confs))

    return text, conf

## 3. DeepSeek-OCR

Load DeepSeek-OCR and provide a helper that returns the OCR text saved by the model.


In [ ]:
import os
import glob
import shutil
import time
import torch
from transformers import AutoModel, AutoTokenizer

MODEL_ID = "deepseek-ai/DeepSeek-OCR"

if "ds_model" not in globals():
    tok = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

    if tok.pad_token is None and tok.eos_token is not None:
        tok.pad_token = tok.eos_token

    ds_model = AutoModel.from_pretrained(
        MODEL_ID,
        trust_remote_code=True,
        use_safetensors=True,
        attn_implementation="eager"
    ).to(dtype=torch.bfloat16, device="cuda").eval()


class OCRTimeoutError(Exception):
    pass


def run_deepseek_ocr(image_path, timeout_seconds=30):
    prompt = "<image>\nFree OCR."
    outdir = "/content/deepseek_tmp"

    if os.path.exists(outdir):
        shutil.rmtree(outdir)
    os.makedirs(outdir, exist_ok=True)

    start_time = time.time()

    with torch.inference_mode():
        ds_model.infer(
            tok,
            prompt=prompt,
            image_file=image_path,
            output_path=outdir,
            base_size=768,
            image_size=512,
            crop_mode=False,
            save_results=True
        )

    if time.time() - start_time > timeout_seconds:
        raise OCRTimeoutError()

    # 1) prefer .mmd
    files = sorted(glob.glob(f"{outdir}/*.mmd"), reverse=True)

    # 2) fallback: any text-like file
    if not files:
        files = sorted(
            glob.glob(f"{outdir}/*.txt") +
            glob.glob(f"{outdir}/*.md") +
            glob.glob(f"{outdir}/*.markdown"),
            reverse=True
        )

    # 3) debug: show what DeepSeek actually saved
    print("  [deepseek] saved files:", os.listdir(outdir))

    if not files:
        return ""

    with open(files[0], "r", encoding="utf-8", errors="ignore") as f:
        content = f.read().strip()

    return content

## 4. OCR Quality and Repetition Detection

Estimate basic text quality and detect repeated output that may indicate an OCR generation loop.


In [ ]:
def estimate_text_quality(text):
    total = len(text)
    if total == 0:
        return {"quality_score": 0}

    alnum = sum(c.isalnum() for c in text)
    noise = sum(1 for c in text if not (c.isalnum() or c.isspace()))

    score = (alnum/total)*70 + (1 - noise/total)*30

    return {"quality_score": round(score, 2)}


def repetition_score(text):
    words = re.findall(r"\w+", text.lower())
    if len(words) < 10:
        return 0.0

    counts = Counter(words)
    return sum(c for _, c in counts.most_common(5)) / len(words)


def is_looping_output(text, threshold=0.4):
    words = re.findall(r"\w+", text.lower())

    if len(words) < 50:
        return False

    counts = Counter(words)
    most_common = counts.most_common(5)

    repeat_ratio = sum(c for _, c in most_common) / len(words)

    # also check line repetition (real loops)
    lines = [l.strip() for l in text.split("\n") if l.strip()]
    unique_lines = len(set(lines))
    line_repeat_ratio = 1 - (unique_lines / max(1, len(lines)))

    # REAL loop condition = both high repetition AND repeated lines
    return (repeat_ratio > threshold) and (line_repeat_ratio > 0.3)

## 5. Hybrid OCR Selection

Use Tesseract first. If its confidence is low or the output appears repetitive, attempt DeepSeek-OCR and fall back to Tesseract when necessary.


In [ ]:
def hybrid_ocr(img_path):
    print("  [hybrid] loading image", flush=True)
    img = cv2.imread(img_path)

    if img is None:
        raise FileNotFoundError(f"Could not read image: {img_path}")

    print("  [hybrid] correcting rotation", flush=True)
    img = correct_rotation(img)

    print("  [hybrid] preprocessing", flush=True)
    _, processed = preprocess_pipeline(img)

    print("  [hybrid] running tesseract", flush=True)
    text_tess, conf_tess = run_tesseract(processed)
    tess_loop = is_looping_output(text_tess)

    print(f"  [hybrid] tesseract conf={conf_tess:.2f}, loop={tess_loop}", flush=True)

    if conf_tess > 60 and not tess_loop:
        return {
            "text": text_tess,
            "model": "tesseract",
            "confidence": conf_tess,
            "deepseek_attempted": False,
            "deepseek_status": "not_needed"
        }

    print("  [hybrid] trying deepseek", flush=True)
    try:
        text_ds = run_deepseek_ocr(img_path)

        if not text_ds.strip():
            return {
                "text": text_tess,
                "model": "fallback_tesseract",
                "confidence": conf_tess,
                "deepseek_attempted": True,
                "deepseek_status": "empty_output"
            }

        ds_loop = is_looping_output(text_ds)
        print(f"  [hybrid] deepseek loop={ds_loop}", flush=True)

        if ds_loop:
            return {
                "text": text_tess,
                "model": "fallback_tesseract",
                "confidence": conf_tess,
                "deepseek_attempted": True,
                "deepseek_status": "looping"
            }

        return {
            "text": text_ds,
            "model": "deepseek",
            "confidence": conf_tess,
            "deepseek_attempted": True,
            "deepseek_status": "accepted"
        }

    except OCRTimeoutError:
        return {
            "text": text_tess,
            "model": "fallback_tesseract",
            "confidence": conf_tess,
            "deepseek_attempted": True,
            "deepseek_status": "timeout"
        }

    except Exception as e:
        return {
            "text": text_tess,
            "model": "fallback_tesseract",
            "confidence": conf_tess,
            "deepseek_attempted": True,
            "deepseek_status": f"error: {str(e)[:80]}"
        }

## 6. Baseline Rule-Based Classification

An early baseline classifier used simple keywords. It is kept here to show the initial approach; the later NLP stage replaces it with Qwen-based classification.


In [ ]:
def classify_text(text):
    text = text.lower()

    if "cancer" in text:
        return "health"
    elif "government" in text:
        return "politics"
    elif "market" in text:
        return "business"
    else:
        return "general"

## 7. Dataset and OCR Result File

Configure the input image folder and create the CSV file used to store OCR results.


In [ ]:
IMAGE_FOLDER = "/content/News"  # Update this path to your scanned-image folder

def get_images(folder):
    return sorted(glob.glob(folder + "/*.jpg"))

image_paths = get_images(IMAGE_FOLDER)

csv_file = open("/content/final_results.csv", "w", newline="", encoding="utf-8")
writer = csv.DictWriter(csv_file, fieldnames=[
    "image_name",
    "model",
    "confidence",
    "quality_score",
    "looping",
    "deepseek_attempted",
    "deepseek_status",
    "text"
])
writer.writeheader()

## 8. Run the OCR Pipeline

Process each image, record the selected OCR model and quality information, and save the extracted text.


In [ ]:
image_paths = get_images(IMAGE_FOLDER)

for i, img_path in enumerate(image_paths, start=1):
    print(f"[{i}/{len(image_paths)}] Processing {os.path.basename(img_path)}")

    result = hybrid_ocr(img_path)

    text = result["text"]
    model_used = result["model"]
    conf = result["confidence"]
    ds_attempted = result["deepseek_attempted"]
    ds_status = result["deepseek_status"]

    metrics = estimate_text_quality(text)
    looping = is_looping_output(text)

    writer.writerow({
        "image_name": os.path.basename(img_path),
        "model": model_used,
        "confidence": conf,
        "quality_score": metrics["quality_score"],
        "looping": looping,
        "deepseek_attempted": ds_attempted,
        "deepseek_status": ds_status,
        "text": text[:2000]
    })

    print(
        f"  -> model={model_used}, conf={conf:.2f}, looping={looping}, "
        f"ds_attempted={ds_attempted}, ds_status={ds_status}"
    )

# Ensure all rows are written before pandas reads the OCR result file.
csv_file.close()
print("OCR stage complete. Results saved to /content/final_results.csv")


## 9. Qwen NLP Processing

Load the OCR results and use Qwen2.5-0.5B-Instruct for summarisation, classification, and metadata extraction.


In [ ]:
import pandas as pd

INPUT_CSV = "/content/final_results.csv"
OUTPUT_CSV = "/content/final_results_with_nlp.csv"

df = pd.read_csv(INPUT_CSV)

print("Loaded rows:", len(df))
df.head()

In [ ]:
!pip install -q transformers accelerate sentencepiece

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

QWEN_MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"

qwen_tokenizer = AutoTokenizer.from_pretrained(QWEN_MODEL_ID, use_fast=True)

qwen_model = AutoModelForCausalLM.from_pretrained(
    QWEN_MODEL_ID,
    device_map="auto",
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32
)

qwen_model.eval()
print("Qwen loaded")

### 9.1 Summarisation

Generate a short factual summary while instructing the model not to add unsupported information.


In [ ]:
def summarize_qwen(text, max_chars=1500):
    text = str(text).strip()
    if not text:
        return ""

    text = text[:max_chars]

    prompt = f"""
Summarize the following news article in 2-3 sentences.

Rules:
- Keep key information only
- Do not repeat
- Do not add new facts

TEXT:
{text}
"""

    messages = [
        {"role": "system", "content": "You summarize news articles."},
        {"role": "user", "content": prompt},
    ]

    enc = qwen_tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_tensors="pt",
        return_dict=True
    )

    enc = {k: v.to(qwen_model.device) for k, v in enc.items()}

    with torch.no_grad():
        output = qwen_model.generate(
            **enc,
            max_new_tokens=120,
            do_sample=False,
            repetition_penalty=1.1,
            eos_token_id=qwen_tokenizer.eos_token_id,
            pad_token_id=qwen_tokenizer.eos_token_id,
        )

    input_len = enc["input_ids"].shape[-1]
    gen = output[0][input_len:]
    return qwen_tokenizer.decode(gen, skip_special_tokens=True).strip()

### 9.2 Qwen-Based Classification

Classify each usable OCR article into one of five news categories.


In [ ]:
def classify_qwen(text, max_chars=1000):
    text = str(text).strip()
    if not text:
        return "unknown"

    text = text[:max_chars]

    prompt = f"""
Classify the following news article into ONE category:

Options:
- health
- politics
- business
- sports
- general

Only output the category word.

TEXT:
{text}
"""

    messages = [
        {"role": "system", "content": "You are a news classifier."},
        {"role": "user", "content": prompt},
    ]

    enc = qwen_tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_tensors="pt",
        return_dict=True
    )

    enc = {k: v.to(qwen_model.device) for k, v in enc.items()}

    with torch.no_grad():
        output = qwen_model.generate(
            **enc,
            max_new_tokens=10,
            do_sample=False,
            eos_token_id=qwen_tokenizer.eos_token_id,
            pad_token_id=qwen_tokenizer.eos_token_id,
        )

    input_len = enc["input_ids"].shape[-1]
    gen = output[0][input_len:]
    label = qwen_tokenizer.decode(gen, skip_special_tokens=True).strip().lower()

    for cat in ["health", "politics", "business", "sports", "general"]:
        if cat in label:
            return cat

    return "general"

### 9.3 Title and Author Extraction

Extract bibliographic metadata only when it is explicitly present in the OCR text.


In [ ]:
def extract_title_author_qwen(text, max_chars=1500):
    text = str(text).strip()
    if not text:
        return {"title": "unclear", "author": "unclear"}

    text = text[:max_chars]

    prompt = f"""
Extract the TITLE and AUTHOR from the following OCR news text.

Rules:
- Use only information explicitly present in the text.
- Do not guess or invent.
- If the title is unclear, output "unclear".
- If the author is unclear, output "unclear".
- Return the result in exactly this format:

Title: ...
Author: ...

TEXT:
{text}
"""

    messages = [
        {"role": "system", "content": "You extract bibliographic metadata from OCR text carefully."},
        {"role": "user", "content": prompt},
    ]

    enc = qwen_tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_tensors="pt",
        return_dict=True
    )

    enc = {k: v.to(qwen_model.device) for k, v in enc.items()}

    with torch.no_grad():
        output = qwen_model.generate(
            **enc,
            max_new_tokens=80,
            do_sample=False,
            eos_token_id=qwen_tokenizer.eos_token_id,
            pad_token_id=qwen_tokenizer.eos_token_id,
        )

    input_len = enc["input_ids"].shape[-1]
    gen = output[0][input_len:]
    result = qwen_tokenizer.decode(gen, skip_special_tokens=True).strip()

    title = "unclear"
    author = "unclear"

    for line in result.splitlines():
        line = line.strip()
        if line.lower().startswith("title:"):
            value = line.split(":", 1)[1].strip()
            if value:
                title = value
        elif line.lower().startswith("author:"):
            value = line.split(":", 1)[1].strip()
            if value:
                author = value

    return {"title": title or "unclear", "author": author or "unclear"}

### 9.4 NLP Input Quality Filter

Skip severely corrupted OCR text so that unreliable input is not passed to the language model.


In [ ]:
import re

def text_is_usable_for_nlp(text):
    text = str(text).strip()
    if not text:
        return False

    # too short
    words = re.findall(r"\w+", text)
    if len(words) < 30:
        return False

    # too many weird characters
    strange = sum(1 for c in text if not (c.isalnum() or c.isspace() or c in ".,;:!?()[]'\"-/%&"))
    strange_ratio = strange / max(len(text), 1)
    if strange_ratio > 0.15:
        return False

    # too much OCR garbage-looking content
    alnum = sum(c.isalnum() for c in text)
    alnum_ratio = alnum / max(len(text), 1)
    if alnum_ratio < 0.45:
        return False

    # very long broken tokens often indicate OCR corruption
    long_broken_tokens = [w for w in text.split() if len(w) > 25]
    if len(long_broken_tokens) > 3:
        return False

    return True

### 9.5 Run NLP Enrichment

Generate summaries, categories, titles and authors for each OCR result and save progress periodically.


In [ ]:
df["summary"] = ""
df["qwen_category"] = ""
df["title"] = ""
df["author"] = ""

for i in range(len(df)):
    text = df.loc[i, "text"]

    print(f"[{i+1}/{len(df)}] Processing...", flush=True)

    if not text_is_usable_for_nlp(text):
        summary = "OCR text too noisy for reliable summarization."
        category = "general"
        title = "unclear"
        author = "unclear"

        print("  skipped: text too noisy", flush=True)
    else:
        try:
            summary = summarize_qwen(text)
            category = classify_qwen(text)
            meta = extract_title_author_qwen(text)

            title = meta["title"]
            author = meta["author"]

        except Exception as e:
            summary = "ERROR"
            category = "general"
            title = "unclear"
            author = "unclear"
            print("  ERROR:", str(e)[:80], flush=True)

    df.loc[i, "summary"] = summary
    df.loc[i, "qwen_category"] = category
    df.loc[i, "title"] = title
    df.loc[i, "author"] = author

    print("  category:", category, flush=True)
    print("  title:", title, flush=True)
    print("  author:", author, flush=True)
    print("  summary:", summary[:100], "\n", flush=True)

    if i % 10 == 0:
        df.to_csv("/content/final_results_with_nlp.csv", index=False)

df.to_csv("/content/final_results_with_nlp.csv", index=False)
print("DONE", flush=True)

### 9.6 Save Final Results

Write the enriched OCR and NLP results to the final CSV file.


In [ ]:
df.to_csv(OUTPUT_CSV, index=False)
print("Saved to:", OUTPUT_CSV)